## Assignment 2: NLP
# Name: Md. Abdullah Al Mamun
# ID: 1000128810

## NLP Based Question-Answer Tool
We have used a local extractive Question Answering pipeline using Hugging Face `transformers` and a pre-trained `distilbert-base-cased-distilled-squad` model.

In [1]:
#dependencies
!pip install -q pypdf transformers torch

import os
import re
import torch
import pypdf
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [13]:
def extract_text_from_pdf(pdf_path):
    if not os.path.exists(pdf_path):
        return f"Error: File {pdf_path} not found."
    reader = pypdf.PdfReader(pdf_path)
    extracted_text = ""
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            text = re.sub(r'[_$%&()\\\.\*]{3,}', '', text)
            extracted_text += f"--- Page {page_num + 1} ---\n{text}\n"
    return extracted_text

In [10]:
# 3. Load Model and Tokenizer
print("Loading local DistilBERT QA Model and Tokenizer directly...")
model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
print("Model loaded successfully!")

Loading local DistilBERT QA Model and Tokenizer directly...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded successfully!


In [11]:
# 4. QA Inference Function
def local_answer_question(pdf_path, question):
    context = extract_text_from_pdf(pdf_path)
    if context.startswith("Error:"):
        return context

    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    answer_start_index = outputs.start_logits.argmax()
    answer_end_index = outputs.end_logits.argmax()

    predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
    answer = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

    start_score = torch.softmax(outputs.start_logits, dim=-1)[0, answer_start_index].item()
    end_score = torch.softmax(outputs.end_logits, dim=-1)[0, answer_end_index].item()
    confidence = start_score * end_score

    if not answer.strip():
        return "Answer not found in context."
    return f"{answer.strip()} (Confidence: {confidence:.4f})"

In [14]:
# 5. Extract data and evaluate
easy_pdf = "/content/Test PDF (Easy).pdf"
hard_pdf = "/content/Test PDF (Hard).pdf"
questions_pdf = "/content/Questions.pdf"

if os.path.exists(questions_pdf):
    questions_content = extract_text_from_pdf(questions_pdf)
    questions = re.findall(r'\d+\.\s*(.*?)(?=\s*\d+\.\s*|\Z)', questions_content, re.DOTALL)

    print(f"\nFound {len(questions)} questions inside {questions_pdf}. Running local evaluations:\n")
    for idx, q in enumerate(questions, 1):
        clean_q = re.sub(r'\s+', ' ', q).strip()
        print(f"Question {idx}: {clean_q}")

        print("  -> Local Answer (Easy PDF):")
        print(f"     {local_answer_question(easy_pdf, clean_q)}\n")

        print("  -> Local Answer (Hard PDF):")
        print(f"     {local_answer_question(hard_pdf, clean_q)}\n")
        print("=" * 60)
else:
    print(f"Questions file not found at {questions_pdf}. Please check your files.")


Found 6 questions inside /content/Questions.pdf. Running local evaluations:

Question 1: What is the capital city of Bangladesh?
  -> Local Answer (Easy PDF):
     Dhaka (Confidence: 0.9702)

  -> Local Answer (Hard PDF):
     Dhaka (Confidence: 0.9737)

Question 2: Which river is considered the lifeline of Bangladesh?
  -> Local Answer (Easy PDF):
     Padma River (Confidence: 0.7391)

  -> Local Answer (Hard PDF):
     Padma River (Confidence: 0.7443)

Question 3: In which year did Bangladesh gain independence?
  -> Local Answer (Easy PDF):
     1971 (Confidence: 0.9947)

  -> Local Answer (Hard PDF):
     1971 (Confidence: 0.9949)

Question 4: What is the official language of Bangladesh?
  -> Local Answer (Easy PDF):
     Bengali (Confidence: 0.9335)

  -> Local Answer (Hard PDF):
     Bengali (Confidence: 0.9435)

Question 5: What is the name of the world’s largest mangrove forest located in Bangladesh?
  -> Local Answer (Easy PDF):
     Sundarbans (Confidence: 0.9217)

  -> Local